<a href="https://colab.research.google.com/github/gianmarcomejia96/data-analytics-portfolio-gianmarcomejia/blob/main/Ejercicio_ruteo_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import requests
from math import radians, sin, cos, asin, sqrt
from itertools import permutations
from datetime import datetime, timedelta

# =========================
# CONFIG
# =========================
INPUT_XLSX = "/content/INPUT.xlsx"
OPER_XLSX  = "/content/OPER.xlsx"
OUT_XLSX   = "/content/OUTPUT_ROUTING.xlsx"

# Filtros
ALCANCE_COL = "ALCANCE FINAL"   # <- confirmado por ti
ALCANCE_OK  = "X"

# Días
DAY_COL = "DIA"  # contiene JUEVES / VIERNES o vacío
DAY_MAP = {"JUEVES":"Jueves", "VIERNES":"Viernes"}  # output

# Capacidad
MAX_VISITS_PER_OPER_TOTAL = 4

# Servicio (encuesta)
SERVICE_MIN = 40

# Ventanas (alternativas)
SHIFT_WINDOWS = [
    ("08:00", "12:00"),  # mañana
    ("13:00", "17:00"),  # tarde
]

# Score (para asignación)
LAMBDA_DIST_KM = 0.06      # penaliza distancia aérea en score (rápido)
BALANCE_PENALTY = 0.25     # penaliza desbalance Thu/Fri

# OSRM (gratis) - proxy tiempos reales
OSRM_TABLE_URL = "https://router.project-osrm.org/table/v1/driving/"
# Nota: si el endpoint público se satura, puedes montar tu propio OSRM más adelante.


In [2]:

# =========================
# Utilidades
# =========================
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return 2*R*asin(sqrt(a))

def time_to_dt(t_str):
    return datetime.strptime(t_str, "%H:%M")

def fmt_hhmmss_from_seconds(sec):
    if sec is None or np.isnan(sec):
        return ""
    sec = int(round(sec))
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    return f"{h:02d}:{m:02d}:{s:02d}"

def fmt_time(dt):
    return dt.strftime("%H:%M:%S")

def osrm_table_durations_seconds(points_lonlat):
    """
    points_lonlat: list of (lon, lat)
    returns: matrix durations in seconds (NxN)
    """
    coords = ";".join([f"{lon:.6f},{lat:.6f}" for lon,lat in points_lonlat])
    url = OSRM_TABLE_URL + coords
    params = {"annotations":"duration"}
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    return np.array(data["durations"], dtype=float)



In [3]:
# =========================
# 1) Cargar INPUT (visitas)
# =========================
vis = pd.read_excel(INPUT_XLSX).copy()

# Solo ALCANCE FINAL = X
vis = vis[vis[ALCANCE_COL].astype(str).str.strip().eq(ALCANCE_OK)].copy()

# Campos mínimos usados del input
need_cols = ["ID_PDV","CUENTA","NOMBRE_PDV","DIRECCION","CIUDAD","LATITUD","LONGITUD","ZONA", DAY_COL]
missing = [c for c in need_cols if c not in vis.columns]
if missing:
    raise ValueError(f"Faltan columnas en INPUT: {missing}")

# visit_id único (PDV físico puede repetirse por cliente)
vis["visit_id"] = vis["ID_PDV"].astype(str) + "_" + vis["CUENTA"].astype(str)

# Día permitido
def allowed_days(x):
    x = str(x).strip().upper()
    if x in ["JUEVES","VIERNES"]:
        return {x}
    return {"JUEVES","VIERNES"}  # si viene vacío -> flexible

vis["allowed_days"] = vis[DAY_COL].apply(allowed_days)

vis = vis.dropna(subset=["LATITUD","LONGITUD"]).copy()
vis["LATITUD"]  = vis["LATITUD"].astype(float)
vis["LONGITUD"] = vis["LONGITUD"].astype(float)

vis["ZONA"] = vis["ZONA"].astype(str).str.strip().str.upper()
vis["location_id"] = pd.factorize(
    vis[["LATITUD","LONGITUD","ZONA"]].astype(str).agg("|".join, axis=1)
)[0] + 1


# =========================
# 2) Cargar OPER (capacidad máquina)
# =========================
ops = pd.read_excel(OPER_XLSX).copy()

# Solo IDENTIDAD=OPER y con geo
ops = ops[ops["IDENTIDAD"].astype(str).str.strip().str.upper().eq("OPER")].copy()
ops = ops.dropna(subset=["LAT","LONG","NOMBRE","COD MKT"]).copy()
ops["LAT"]  = ops["LAT"].astype(float)
ops["LONG"] = ops["LONG"].astype(float)

# IDs
ops["oper_id"] = ops["DNI"].astype(str)  # o COD MKT si prefieres
ops["oper_name"] = ops["NOMBRE"].astype(str)
ops["cod_mkt"] = ops["COD MKT"]

# Capacidad total del sistema
max_total = len(ops) * MAX_VISITS_PER_OPER_TOTAL
vis = vis.head(max_total).copy()  # opcional: si sobran muchísimas visitas, recortas por algún ranking



In [4]:
# =========================
# 3) Asignación greedy global
# (cobertura + cercanía + balance Thu/Fri)
# =========================
op_ids = ops["oper_id"].tolist()
O = len(op_ids)

vi_ids = vis["visit_id"].tolist()
V = len(vi_ids)

op_lat = ops["LAT"].to_numpy()
op_lon = ops["LONG"].to_numpy()
vi_lat = vis["LATITUD"].to_numpy()
vi_lon = vis["LONGITUD"].to_numpy()

# score rápido con distancia aérea
pairs = []
for oi in range(O):
    for vi in range(V):
        dkm = haversine_km(op_lat[oi], op_lon[oi], vi_lat[vi], vi_lon[vi])
        s = 1.0 - (LAMBDA_DIST_KM * dkm)
        pairs.append((s, oi, vi, dkm))
pairs.sort(reverse=True, key=lambda x: x[0])

assigned_visit = np.zeros(V, dtype=bool)
oper_total = np.zeros(O, dtype=int)
oper_thu = np.zeros(O, dtype=int)
oper_fri = np.zeros(O, dtype=int)

assignments = []  # (oi, vi, day)

for s, oi, vi, dkm in pairs:
    if assigned_visit[vi]:
        continue
    if oper_total[oi] >= MAX_VISITS_PER_OPER_TOTAL:
        continue

    allowed = vis.iloc[vi]["allowed_days"]

    # elige el día que menos desbalancee
    cand_days = []
    if "JUEVES" in allowed:
        cand_days.append("JUEVES")
    if "VIERNES" in allowed:
        cand_days.append("VIERNES")

    # intenta el día que reduce |thu-fri|, si empate, el que tenga menos carga
    best_day = None
    best_pen = None
    for day in cand_days:
        thu = oper_thu[oi] + (1 if day=="JUEVES" else 0)
        fri = oper_fri[oi] + (1 if day=="VIERNES" else 0)
        pen = BALANCE_PENALTY * abs(thu - fri)
        if best_pen is None or pen < best_pen:
            best_pen = pen
            best_day = day

    if best_day is None:
        continue

    assigned_visit[vi] = True
    oper_total[oi] += 1
    if best_day == "JUEVES":
        oper_thu[oi] += 1
    else:
        oper_fri[oi] += 1

    assignments.append((oi, vi, best_day))


In [ ]:
# =========================
# 4) Construir rutas por oper/día con OSRM + ventanas
# =========================
def schedule_best(oper_row, day_vis_df):
    """
    day_vis_df: visitas asignadas a un oper en un día (<=4)
    Devuelve df con orden, dist, travel_time, arrival, depart, y ruta factible
    """
    if day_vis_df.empty:
        return day_vis_df.assign(
            seq=np.nan, dist_km=np.nan,
            travel_sec=np.nan,
            arrival="", depart="",
            shift=""
        )

    base = (float(oper_row["LONG"]), float(oper_row["LAT"]))  # (lon, lat)

    # puntos: base + visitas
    pts = [base] + list(zip(day_vis_df["LONGITUD"].astype(float), day_vis_df["LATITUD"].astype(float)))
    D = osrm_table_durations_seconds(pts)  # NxN durations

    n = len(day_vis_df)

    # indices de visitas dentro de la matriz (1..n)
    idxs = list(range(1, n+1))

    best = None
    best_total_sec = None
    best_sched = None
    best_shift = None

    # prueba todas las permutaciones (<= 24)
    for perm in permutations(idxs, len(idxs)):
        # para cada ventana, intenta schedule
        for sh_start, sh_end in SHIFT_WINDOWS:
            t = time_to_dt(sh_start)
            t_end = time_to_dt(sh_end)

            cur = 0  # base index
            ok = True
            arrivals = []
            departs = []
            leg_secs = []
            leg_kms = []  # aproximación por haversine (solo para "Distancia" visible)

            for p in perm:
                travel_sec = D[cur, p]
                if np.isnan(travel_sec) or travel_sec <= 0:
                    ok = False
                    break
                t = t + timedelta(seconds=float(travel_sec))
                # llegada
                arrivals.append(fmt_time(t))
                # servicio
                t = t + timedelta(minutes=SERVICE_MIN)
                departs.append(fmt_time(t))

                leg_secs.append(float(travel_sec))

                # dist km visible (haversine entre puntos)
                lon1, lat1 = pts[cur]
                lon2, lat2 = pts[p]
                leg_kms.append(haversine_km(lat1, lon1, lat2, lon2))

                # chequeo ventana (todo debe terminar antes del fin)
                if t > t_end:
                    ok = False
                    break

                cur = p

            if not ok:
                continue

            total_service_sec = sum(int(day_vis_df.iloc[i-1].get("n_visits", 1)) for i in perm) * (SERVICE_MIN*60)
            total_sec = sum(leg_secs) + total_service_sec
            if best_total_sec is None or total_sec < best_total_sec:
                best_total_sec = total_sec
                best = perm
                best_sched = (arrivals, departs, leg_secs, leg_kms)
                best_shift = f"{sh_start}-{sh_end}"

    if best is None:
        # no factible en mañana/tarde -> igual devolvemos un orden por cercanía, pero marcado
        # (para que veas qué quedó fuera de ventana)
        # orden greedy simple desde base
        remaining = set(idxs)
        cur = 0
        order = []
        while remaining:
            nxt = min(remaining, key=lambda j: D[cur, j])
            order.append(nxt)
            remaining.remove(nxt)
            cur = nxt
        best = tuple(order)
        best_shift = "NO_FIT"

        # arma schedule sin cortar (solo para mostrar)
        t = time_to_dt(SHIFT_WINDOWS[0][0])
        arrivals, departs, leg_secs, leg_kms = [], [], [], []
        cur = 0
        for p in best:
            travel_sec = float(D[cur, p])
            t = t + timedelta(seconds=travel_sec)
            arrivals.append(fmt_time(t))
            service_minutes = int(day_vis_df.iloc[p-1].get("n_visits", 1)) * SERVICE_MIN
            t = t + timedelta(minutes=service_minutes)
            departs.append(fmt_time(t))
            leg_secs.append(travel_sec)
            lon1, lat1 = pts[cur]
            lon2, lat2 = pts[p]
            leg_kms.append(haversine_km(lat1, lon1, lat2, lon2))
            cur = p
        best_sched = (arrivals, departs, leg_secs, leg_kms)

    arrivals, departs, leg_secs, leg_kms = best_sched

    # mapear orden a filas
    ordered_vis = day_vis_df.copy()
    # best contiene indices 1..n en el orden de visita
    location_ids_in_order = [day_vis_df.iloc[i-1]["location_id"] for i in best]
    rank = {loc_id: k+1 for k, loc_id in enumerate(location_ids_in_order)}
    ordered_vis["seq"] = ordered_vis["location_id"].map(rank)

    # por cada stop, leg time/km desde punto anterior (base o visita previa)
    # alineamos con el orden
    ordered_vis = ordered_vis.sort_values("seq").reset_index(drop=True)
    ordered_vis["travel_sec"] = leg_secs
    ordered_vis["dist_km"] = leg_kms
    ordered_vis["arrival"] = arrivals
    ordered_vis["depart"] = departs
    ordered_vis["shift"] = best_shift
    return ordered_vis

# construir df asignado
ass = []
for oi, vi, day in assignments:
    row = vis.iloc[vi].to_dict()
    row["oper_id"] = ops.iloc[oi]["oper_id"]
    row["oper_name"] = ops.iloc[oi]["oper_name"]
    row["cod_mkt"] = ops.iloc[oi]["cod_mkt"]
    row["day"] = day
    ass.append(row)

ass = pd.DataFrame(ass)
if ass.empty:
    raise ValueError("No se asignó ninguna visita. Revisa filtros / datos.")

# stops = ubicaciones únicas por oper/día (pero guardamos cuántas visitas hay dentro)
stops = (ass
    .groupby(["oper_id","day","location_id"], as_index=False)
    .agg(
        LATITUD=("LATITUD","first"),
        LONGITUD=("LONGITUD","first"),
        ZONA=("ZONA","first"),
        n_visits=("visit_id","count"),
        visit_ids=("visit_id", lambda s: list(s)),
        oper_name=("oper_name","first"),
        cod_mkt=("cod_mkt","first"),
    )
)

routed_parts = []
for (oper_id, day), g in stops.groupby(["oper_id","day"], sort=False):
    oper_row = ops[ops["oper_id"]==oper_id].iloc[0]
    routed_parts.append(schedule_best(oper_row, g))

routed = pd.concat(routed_parts, ignore_index=True)

In [ ]:
# =========================
# 5) Formato EXACTO del output del especialista
# =========================
OUT_COLS = [
    'ID de territorio',
    'COD MKT',
    'NOMBRE',
    'DIA DE VISITA',
    'Numero de secuencia',
    'Semanas de entrega',
    'Semana de entrega',
    'Frecuencia',
    'ZONA',
    'ID de ubicación',
    'ID PDV',
    'NOMBRE_PDV',
    'Direccion(linea 1)                               ',
    'Municipio                    ',
    'Estado            ',
    'Pais',
    'Longitud  ',
    'Latitud   ',
    'ID de tipo de cuenta',
    'Distancia',
    'Tiempo de desplazamiento',
    'Tiempo de servicio',
    'Hora de llegada del conductor',
    'Conductor de hora de salida',
    'Promotor',
    'Telefono'
]

# ID de ubicación estable (similar a 50xx del ejemplo)
# (si quieres EXACTAMENTE igual a tu especialista, lo ajusto cuando me digas la regla)
routed = routed.reset_index(drop=True)
routed["id_ubic"] = 5000 + routed["location_id"].astype(int)

out = pd.DataFrame({
    'ID de territorio': 1,
    'COD MKT': routed["cod_mkt"].astype(int, errors="ignore"),
    'NOMBRE': routed["oper_name"],
    'DIA DE VISITA': routed["day"].map({"JUEVES":"Jueves","VIERNES":"Viernes"}),
    'Numero de secuencia': routed["seq"].astype(int),
    'Semanas de entrega': 1,
    'Semana de entrega': 1,
    'Frecuencia': "Semanal",
    'ZONA': routed["ZONA"].astype(str),
    'ID de ubicación': routed["id_ubic"].astype(int),
    'ID PDV': routed["ID_PDV"],
    'NOMBRE_PDV': routed["NOMBRE_PDV"],
    'Direccion(linea 1)                               ': routed["DIRECCION"].astype(str),
    'Municipio                    ': routed["CIUDAD"].astype(str),
    'Estado            ': "Lima",
    'Pais': "Peru",
    'Longitud  ': routed["LONGITUD"].astype(float),
    'Latitud   ': routed["LATITUD"].astype(float),
    'ID de tipo de cuenta': "PE_ PHP PG",
    'Distancia': routed["dist_km"].round(2),
    'Tiempo de desplazamiento': routed["travel_sec"].apply(fmt_hhmmss_from_seconds),
    'Tiempo de servicio': "00:40:00",
    'Hora de llegada del conductor': routed["arrival"],
    'Conductor de hora de salida': routed["depart"],
    'Promotor': routed["oper_name"],
    'Telefono': ""  # <- confirmado: no lo necesitas
})

# asegurar orden de columnas EXACTO
out = out[OUT_COLS]



In [ ]:
# =========================
# 6) Export
# =========================
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
    out.to_excel(w, index=False, sheet_name="BD Master_Ejercicio 3 Routing")
    # opcional: hoja de control
    kpi = pd.DataFrame([{
        "opers": len(ops),
        "visitas_input_alcanceX": len(vis),
        "visitas_asignadas": len(out),
        "capacidad_total": len(ops)*MAX_VISITS_PER_OPER_TOTAL,
        "service_min": SERVICE_MIN,
        "shift_windows": str(SHIFT_WINDOWS),
        "osrm": OSRM_TABLE_URL
    }])
    kpi.to_excel(w, index=False, sheet_name="KPIs")

print("OK ->", OUT_XLSX)
print(out.head(3))


OK -> /content/OUTPUT_ROUTING.xlsx
   ID de territorio  COD MKT                        NOMBRE DIA DE VISITA  \
0                 1    14068  AVILA SANTIVAÑEZ JUAN MARTIN        Jueves   
1                 1    14068  AVILA SANTIVAÑEZ JUAN MARTIN        Jueves   
2                 1    14068  AVILA SANTIVAÑEZ JUAN MARTIN        Jueves   

   Numero de secuencia  Semanas de entrega  Semana de entrega Frecuencia  \
0                    1                   1                  1    Semanal   
1                    2                   1                  1    Semanal   
2                    3                   1                  1    Semanal   

     ZONA  ID de ubicación  ... Longitud   Latitud    ID de tipo de cuenta  \
0   OESTE             5001  ... -77.073307 -12.090107           PE_ PHP PG   
1  CENTRO             5002  ... -77.072536 -12.090631           PE_ PHP PG   
2   OESTE             5003  ... -77.072669 -12.091491           PE_ PHP PG   

  Distancia Tiempo de desplazamiento Tiemp